# RAG: Baseline (Bi-Encoder) vs. Re-Ranking (Cross-Encoder)

**Document:** IFAB *Laws of the Game 2026/27* (single-page edition, 260 pages, effective 1 July 2026).

**Pipelines (same chunked corpus for both):**
1. **Baseline** – bi-encoder (`all-MiniLM-L6-v2`) + FAISS → Top 10 → Top 3 straight to the LLM → *Answer 1*
2. **Re-rank** – same Top 10 → cross-encoder (`ms-marco-MiniLM-L-6-v2`) → re-sorted Top 3 → same LLM prompt → *Answer 2*

The LLM (Gemini) is instructed to answer **only** from the retrieved chunks, so any difference between Answer 1 and Answer 2 comes from retrieval, not from the model's own football knowledge.

## 1. Setup

In [1]:
# Run once if packages are missing (CPU-only torch is enough for these small models):
# !pip install torch --index-url https://download.pytorch.org/whl/cpu
# !pip install pymupdf sentence-transformers faiss-cpu google-genai python-dotenv pandas numpy

In [2]:
import os, re, json, time, hashlib, textwrap, urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
import pymupdf                      # PDF parsing (PyMuPDF)
import faiss                        # vector store
from sentence_transformers import SentenceTransformer, CrossEncoder
from google import genai
from google.genai import types
from google.genai import errors as genai_errors
from dotenv import load_dotenv

pd.set_option("display.max_colwidth", 120)

# ---------------------------------------------------------------------------
# API KEY: put your Gemini key in a file called `.env` next to this notebook:
#     GEMINI_API_KEY=your-key-here
# (.env is git-ignored so the key never ends up on GitHub.)
# Alternatively: os.environ["GEMINI_API_KEY"] = "your-key-here"
# ---------------------------------------------------------------------------
load_dotenv()

PDF_URL  = "https://www.theifab.com/downloads/laws-of-the-game-202627-single-pages?l=en"
PDF_PATH = Path("data/LOTG_2026-27_single.pdf")

BI_ENCODER    = "sentence-transformers/all-MiniLM-L6-v2"
CROSS_ENCODER = "cross-encoder/ms-marco-MiniLM-L-6-v2"
LLM_MODEL     = "gemini-3.5-flash-lite"   # free tier: flash allows only 20 requests/day; flash-lite has more headroom

TOP_K_RETRIEVE = 10   # candidates from the bi-encoder
TOP_K_CONTEXT  = 3    # chunks passed to the LLM

/home/neill-rawani/Documents/Assignments/LMA/Assignment/RAG_Reranking/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Load the PDF
The booklet may not be redistributed, so it is downloaded from the IFAB site instead of being committed to the repo.

In [3]:
if not PDF_PATH.exists():
    PDF_PATH.parent.mkdir(exist_ok=True)
    req = urllib.request.Request(PDF_URL, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(req) as r, open(PDF_PATH, "wb") as f:
        f.write(r.read())

doc = pymupdf.open(PDF_PATH)
print(f"{PDF_PATH.name}: {doc.page_count} pages")

LOTG_2026-27_single.pdf: 260 pages


## 3. Parsing
- Page numbers in the booklet match PDF page numbers, so each page is tagged with its section using the table of contents (pp. 6–7).
- Cover, contents and divider/photo pages are skipped (< 15 words).
- The running footer and page number at the bottom of every page are dropped.
- PyMuPDF leaves some layout characters (`\x07`, en-spaces) which are cleaned out.

In [4]:
# (start page, section name) taken from the booklet's contents page
SECTIONS = [
    (9,   "About the Laws"),
    (16,  "Notes on the Laws of the Game"),
    (20,  "General modifications"),
    (22,  "Time-limited substitution protocol"),
    (24,  "Off-field treatment and assessment protocol"),
    (26,  "Throw-in and goal-kick countdown protocol"),
    (28,  "'Only the captain' guidelines"),
    (32,  "Guidelines for temporary dismissals (sin bins)"),
    (36,  "Guidelines for return substitutes"),
    (38,  "Additional permanent concussion substitutions protocol"),
    (45,  "Law 1 - The Field of Play"),
    (55,  "Law 2 - The Ball"),
    (59,  "Law 3 - The Players"),
    (67,  "Law 4 - The Players' Equipment"),
    (73,  "Law 5 - The Referee"),
    (85,  "Law 6 - The Other Match Officials"),
    (93,  "Law 7 - The Duration of the Match"),
    (97,  "Law 8 - The Start and Restart of Play"),
    (101, "Law 9 - The Ball in and out of Play"),
    (103, "Law 10 - Determining the Outcome of a Match"),
    (109, "Law 11 - Offside"),
    (115, "Law 12 - Fouls and Misconduct"),
    (131, "Law 13 - Free Kicks"),
    (135, "Law 14 - The Penalty Kick"),
    (141, "Law 15 - The Throw-in"),
    (145, "Law 16 - The Goal Kick"),
    (149, "Law 17 - The Corner Kick"),
    (152, "Video assistant referee (VAR) protocol"),
    (162, "FIFA Quality Programme"),
    (166, "Law changes 2026/27 - Outline summary"),
    (170, "Law changes 2026/27 - Details of all Law changes"),
    (188, "Law changes 2026/27 - Editorial changes"),
    (194, "Glossary"),
    (208, "Practical guidelines for match officials"),
]
FIRST_PAGE, LAST_PAGE = 9, 257   # everything before p.9 is cover/contents, after p.257 is blank


def section_for(page_no):
    name = None
    for start, sec in SECTIONS:
        if page_no >= start:
            name = sec
    return name


def clean(text):
    text = text.replace("\x07", "").replace("\u2002", " ").replace("\t", " ").replace("\u00ad", "")
    text = re.sub(r"-\n(?=[a-z])", "", text)   # re-join words hyphenated across lines
    text = re.sub(r"\s*\n\s*", " ", text)      # a PDF block = one paragraph
    return re.sub(r"\s{2,}", " ", text).strip()


def split_long(text, limit=150):
    """Split an unusually long paragraph into groups of sentences of <= limit words."""
    if len(text.split()) <= limit:
        return [text]
    out, cur = [], ""
    for sent in re.split(r"(?<=[.;])\s+", text):
        if cur and len((cur + " " + sent).split()) > limit:
            out.append(cur)
            cur = sent
        else:
            cur = (cur + " " + sent).strip()
    return out + [cur] if cur else out


def extract_paragraphs(doc):
    paras = []
    for page_no in range(FIRST_PAGE, LAST_PAGE + 1):
        page = doc[page_no - 1]
        if len(page.get_text().split()) < 15:          # divider / photo pages
            continue
        height = page.rect.height
        for x0, y0, x1, y1, text, *_ in page.get_text("blocks", sort=True):
            if y0 > height - 40:                       # footer + page number
                continue
            text = clean(text)
            if not text or text.isdigit():
                continue
            for piece in split_long(text):
                paras.append({"page": page_no, "section": section_for(page_no), "text": piece})
    return paras


paragraphs = extract_paragraphs(doc)
print(f"{len(paragraphs)} paragraphs extracted")

1578 paragraphs extracted


## 4. Chunking
Paragraphs are packed into chunks of up to ~160 words, never crossing a section boundary, with ~40 words of overlap
(the last paragraph(s) of a chunk are repeated at the start of the next one).
160 words keeps most chunks inside MiniLM's 256-token input limit.

The section name is prepended to the text that gets embedded (e.g. `Law 11 - Offside: ...`) so a chunk still makes sense out of context.

In [5]:
MAX_WORDS, OVERLAP_WORDS = 160, 40


def is_heading(text):
    return len(text.split()) <= 8 and not text.endswith((".", ":", ";"))


def build_chunks(paras, max_words=MAX_WORDS, overlap_words=OVERLAP_WORDS):
    by_section = {}
    for p in paras:
        by_section.setdefault(p["section"], []).append(p)

    chunks = []
    for section, ps in by_section.items():
        cur, cur_words = [], 0

        def flush():
            if cur and not all(is_heading(c["text"]) for c in cur):
                chunks.append({"section": section,
                               "pages": sorted({c["page"] for c in cur}),
                               "text": " ".join(c["text"] for c in cur)})

        for p in ps:
            n = len(p["text"].split())
            # close the chunk when full (but never right after a heading)
            if cur and cur_words + n > max_words and not is_heading(cur[-1]["text"]):
                flush()
                carry, w = [], 0
                for c in reversed(cur):                # overlap
                    cw = len(c["text"].split())
                    if w + cw > overlap_words:
                        break
                    carry.insert(0, c)
                    w += cw
                cur, cur_words = carry, w
            cur.append(p)
            cur_words += n
        flush()

    for i, c in enumerate(chunks):
        c["id"] = i
        c["embed_text"] = f"{c['section']}: {c['text']}"
    return chunks


chunks = build_chunks(paragraphs)
chunk_df = pd.DataFrame(chunks)
chunk_df["words"] = chunk_df["text"].str.split().str.len()
print(f"{len(chunks)} chunks | words per chunk: min {chunk_df.words.min()}, "
      f"mean {chunk_df.words.mean():.0f}, max {chunk_df.words.max()}")
chunk_df[["id", "section", "pages", "words", "text"]].sample(5, random_state=0)

301 chunks | words per chunk: min 41, mean 142, max 271


,id,section,pages,words,text
223,223,Law changes 2026/27 - Details of all Law changes,"[170, 171]",136,Explanation Teams are now permitted to use eight named substitutes in senior ‘A’ international team friendly matches...
150,150,Law 12 - Fouls and Misconduct,[120],136,Advantage If the referee plays the advantage for an offence for which a caution/sending-off would have been issued h...
226,226,Law changes 2026/27 - Details of all Law changes,[173],186,Law 4 – The Players’ Equipment (p. 67) 1. Safety Amended text A player must not use equipment or wear anything that ...
296,296,Practical guidelines for match officials,"[235, 236]",179,"An attacker in an offside position (A) runs towards the ball, preventing the opponent (B) from playing or being able..."
52,52,Law 1 - The Field of Play,[49],159,9. The technical area The technical area relates to matches played in stadiums with a designated sitting area for te...


## 5. Pipeline 1 – Bi-Encoder index (FAISS)

In [6]:
bi_encoder = SentenceTransformer(BI_ENCODER, device="cpu")

t0 = time.perf_counter()
embeddings = bi_encoder.encode([c["embed_text"] for c in chunks], batch_size=32,
                               normalize_embeddings=True, show_progress_bar=True)
print(f"Embedded {len(chunks)} chunks in {time.perf_counter() - t0:.1f}s, dim = {embeddings.shape[1]}")

# Vectors are L2-normalised, so inner product == cosine similarity
index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings.astype("float32"))


def retrieve(query, k=TOP_K_RETRIEVE):
    """Baseline: top-k chunks by cosine similarity. Returns list of (chunk_id, score)."""
    q = bi_encoder.encode([query], normalize_embeddings=True).astype("float32")
    scores, ids = index.search(q, k)
    return list(zip(ids[0].tolist(), scores[0].tolist()))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2855.27it/s]

Batches:   0%|          | 0/10 [00:00<?, ?it/s]

Batches:  10%|█         | 1/10 [00:01<00:09,  1.07s/it]

Batches:  20%|██        | 2/10 [00:02<00:09,  1.16s/it]

...

Batches:  80%|████████  | 8/10 [00:07<00:01,  1.27it/s]

Batches:  90%|█████████ | 9/10 [00:07<00:00,  1.44it/s]

Batches: 100%|██████████| 10/10 [00:07<00:00,  1.94it/s]

Batches: 100%|██████████| 10/10 [00:07<00:00,  1.26it/s]

Embedded 301 chunks in 8.0s, dim = 384


## 6. Pipeline 2 – Cross-Encoder re-ranker

In [7]:
cross_encoder = CrossEncoder(CROSS_ENCODER, device="cpu")


def rerank(query, candidates):
    """Score every (query, chunk) pair jointly and re-sort.
    Returns list of (chunk_id, ce_score, original_rank)."""
    pairs = [(query, chunks[cid]["embed_text"]) for cid, _ in candidates]
    scores = cross_encoder.predict(pairs)
    order = np.argsort(-scores)
    return [(candidates[j][0], float(scores[j]), j + 1) for j in order]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 3351.05it/s]

## 7. LLM + grounded prompt
The same prompt template is used for both pipelines; only the three context chunks differ.

In [8]:
client = genai.Client(api_key=os.environ.get("GEMINI_API_KEY"))

SYSTEM_PROMPT = """You are answering questions about the IFAB Laws of the Game 2026/27.
Rules:
- Use ONLY the context passages provided. Do NOT use any prior knowledge of football rules,
  even if you believe the context is incomplete or the rules used to be different.
- If the context does not contain the answer, reply exactly: "Not found in the provided context."
- Cite the passage numbers you used, e.g. [C2].
- Be concise (max 4 sentences)."""


def build_prompt(query, chunk_ids):
    context = "\n\n".join(
        f"[C{n}] ({chunks[cid]['section']}, p. {chunks[cid]['pages'][0]})\n{chunks[cid]['text']}"
        for n, cid in enumerate(chunk_ids, 1))
    return f"Context:\n{context}\n\nQuestion: {query}\nAnswer:"


# The free tier allows only ~5 requests/minute, so calls are spaced out, retried on 429,
# and cached to disk (re-running the notebook does not spend quota on the same prompt twice).
CACHE_PATH = Path("llm_cache.json")
llm_cache = json.loads(CACHE_PATH.read_text()) if CACHE_PATH.exists() else {}
MIN_SECONDS_BETWEEN_CALLS = 13
_last_call = [0.0]
llm_seconds = [None]


def generate(query, chunk_ids, use_cache=True):
    prompt = build_prompt(query, chunk_ids)
    key = hashlib.sha256(f"{LLM_MODEL}|{SYSTEM_PROMPT}|{prompt}".encode()).hexdigest()
    if use_cache and key in llm_cache:
        return llm_cache[key]

    for attempt in range(6):
        time.sleep(max(0, MIN_SECONDS_BETWEEN_CALLS - (time.perf_counter() - _last_call[0])))
        _last_call[0] = time.perf_counter()
        try:
            t0 = time.perf_counter()
            resp = client.models.generate_content(
                model=LLM_MODEL,
                contents=prompt,
                config=types.GenerateContentConfig(system_instruction=SYSTEM_PROMPT, temperature=0),
            )
            llm_seconds[0] = time.perf_counter() - t0      # API time only (excludes rate-limit waits)
            break
        except genai_errors.APIError as e:
            # 429 = rate limit, 500/503 = model temporarily overloaded -> wait and retry
            if e.code not in (429, 500, 503) or attempt == 5:
                raise
            time.sleep(60 if e.code == 429 else 20)

    llm_cache[key] = resp.text.strip()
    CACHE_PATH.write_text(json.dumps(llm_cache, indent=1))
    return llm_cache[key]

## 8. Running both pipelines

In [9]:
def run_pipelines(query, with_llm=True):
    """Runs baseline and re-rank pipelines for one query, timing each stage."""
    t0 = time.perf_counter()
    base = retrieve(query)                                   # bi-encoder top 10
    t_retrieve = time.perf_counter() - t0

    t0 = time.perf_counter()
    reranked = rerank(query, base)                           # cross-encoder over the same 10
    t_rerank = time.perf_counter() - t0

    base_top3 = [cid for cid, _ in base[:TOP_K_CONTEXT]]
    rr_top3   = [cid for cid, _, _ in reranked[:TOP_K_CONTEXT]]

    out = {"query": query, "base": base, "reranked": reranked,
           "base_top3": base_top3, "rr_top3": rr_top3,
           "t_retrieve": t_retrieve, "t_rerank": t_rerank}

    if with_llm:
        out["answer1"] = generate(query, base_top3)
        out["answer2"] = generate(query, rr_top3)
    return out


def rank_table(res):
    """Side-by-side view of the 10 candidates: baseline rank vs cross-encoder rank."""
    ce_rank = {cid: r for r, (cid, _, _) in enumerate(res["reranked"], 1)}
    ce_score = {cid: s for cid, s, _ in res["reranked"]}
    rows = [{"chunk": cid, "section": chunks[cid]["section"], "page": chunks[cid]["pages"][0],
             "bi_rank": r, "bi_score": round(s, 3),
             "ce_rank": ce_rank[cid], "ce_score": round(ce_score[cid], 2),
             "shift": r - ce_rank[cid], "preview": chunks[cid]["text"][:90]}
            for r, (cid, s) in enumerate(res["base"], 1)]
    return pd.DataFrame(rows)


def ask(query, with_llm=True):
    res = run_pipelines(query, with_llm)
    print("Q:", query)
    display(rank_table(res))
    if with_llm:
        print("\nAnswer 1 (baseline top 3: %s):" % res["base_top3"])
        print(textwrap.fill(res["answer1"], 110))
        print("\nAnswer 2 (re-ranked top 3: %s):" % res["rr_top3"])
        print(textwrap.fill(res["answer2"], 110))
    return res


_ = ask("Can a goal be scored directly from a kick-off?", with_llm=False)

Q: Can a goal be scored directly from a kick-off?


,chunk,section,page,bi_rank,bi_score,ce_rank,ce_score,shift,preview
0,194,Law 16 - The Goal Kick,146,1,0.658,6,2.74,-5,If a player enters the penalty area before the ball is in play and fouls or is fouled by a
1,191,Law 16 - The Goal Kick,145,2,0.654,3,7.87,-1,"The Goal Kick A goal kick is awarded when the whole of the ball passes over the goal line,"
2,177,Law 14 - The Penalty Kick,135,3,0.624,2,8.07,1,The Penalty Kick A penalty kick is awarded if a player commits a direct free kick offence
3,124,Law 10 - Determining the Outcome of a Match,103,4,0.623,4,4.04,0,Determining the Outcome of a Match 1. Goal scored A goal is scored when the whole of the b
4,119,Law 8 - The Start and Restart of Play,97,5,0.616,1,8.20,4,"For every kick-off: • all players, except the player taking the kick-off, must be in their"
5,192,Law 16 - The Goal Kick,145,6,0.612,8,0.55,-2,2. Offences and sanctions If a player is unfairly delaying the taking of a goal kick for t
6,241,Law changes 2026/27 - Details of all Law changes,184,7,0.604,7,1.54,0,Explanation Text introduced to avoid delays at goal kicks. If a team is deliberately delay
7,182,Law 14 - The Penalty Kick,137,8,0.597,5,3.43,3,except for the following when play will be stopped and restarted with an indirect free kic
8,138,Law 11 - Offside,112,9,0.597,10,-0.92,-1,An attacking player may step or stay off the field of play not to be involved in active pl
9,193,Law 16 - The Goal Kick,145,10,0.590,9,0.33,1,"If, after the ball is in play, the kicker touches the ball again before it has touched ano"


## 9. Failure-mode queries
Each query has a `gold` regex that identifies the chunk containing the decisive rule, so we can track where it ranks
before and after re-ranking. (Draft list, to be refined.)

In [10]:
FAILURE_QUERIES = [
    {"type": "Keyword trap",
     "query": "Can a goal be scored directly from a kick-off?",
     "gold": r"goal may be scored directly against the opponents from the kick-off"},
    {"type": "Exception / corner case",
     "query": "What happens if the ball goes straight into the kicker's own goal from the kick-off?",
     "gold": r"kick-off; if the ball directly enters the kicker.s goal"},
    {"type": "Negation / role limit",
     "query": "Can a replaced goalkeeper take a kick in the penalty shoot-out?",
     "gold": r"replaced goalkeeper takes no further part and may not take a kick"},
    {"type": "Buried chunk",
     "query": "When does the referee's authority to show cards begin and end?",
     "gold": r"power to show yellow or red cards"},
    {"type": "Negation",
     "query": "When is a player in an offside position not penalised for being offside?",
     "gold": r"It is not an offence to be in an offside position"},
    {"type": "Exception",
     "query": "What happens if a team has fewer than seven players during penalties?",
     "gold": r"must not abandon the match if a team is reduced to fewer than seven"},
    {"type": "New 2026/27 rule (grounding check)",
     "query": "What happens if the goalkeeper holds the ball in their hands for too long?",
     "gold": r"more than eight seconds before releasing"},
    {"type": "Document-specific: struck-through (deleted) text extracted as live text",
     "query": "Is the kicker cautioned if both the penalty taker and the goalkeeper commit an offence at the same time?",
     "gold": r"kicker is cautioned"},
]


def gold_ids(pattern):
    return {c["id"] for c in chunks if re.search(pattern, c["text"], re.I | re.S)}


def best_rank(ranked_ids, gold):
    ranks = [r for r, cid in enumerate(ranked_ids, 1) if cid in gold]
    return min(ranks) if ranks else None    # None = not in the top 10


def rank_shift_summary(with_llm=False):
    rows, results = [], []
    for fq in FAILURE_QUERIES:
        res = run_pipelines(fq["query"], with_llm)
        gold = gold_ids(fq["gold"])
        b = best_rank([cid for cid, _ in res["base"]], gold)
        c = best_rank([cid for cid, _, _ in res["reranked"]], gold)
        rows.append({"type": fq["type"], "query": fq["query"], "gold_chunks": sorted(gold),
                     "bi_rank": b, "ce_rank": c,
                     "in_top3_baseline": b is not None and b <= TOP_K_CONTEXT,
                     "in_top3_rerank": c is not None and c <= TOP_K_CONTEXT})
        results.append(res)
    return pd.DataFrame(rows), results


summary_df, _ = rank_shift_summary(with_llm=False)
summary_df

,type,query,gold_chunks,bi_rank,ce_rank,in_top3_baseline,in_top3_rerank
0,Keyword trap,Can a goal be scored directly from a kick-off?,[119],5,1,False,True
1,Exception / corner case,What happens if the ball goes straight into the kicker's own goal from the kick-off?,[119],9,1,False,True
2,Negation / role limit,Can a replaced goalkeeper take a kick in the penalty shoot-out?,[126],6,1,False,True
3,Buried chunk,When does the referee's authority to show cards begin and end?,"[31, 94]",8,1,False,True
4,Negation,When is a player in an offside position not penalised for being offside?,[131],4,1,False,True
5,Exception,What happens if a team has fewer than seven players during penalties?,[130],2,1,True,True
6,New 2026/27 rule (grounding check),What happens if the goalkeeper holds the ball in their hands for too long?,[147],2,1,True,True
7,Document-specific: struck-through (deleted) text extracted as live text,Is the kicker cautioned if both the penalty taker and the goalkeeper commit an offence at the same time?,"[128, 234, 238]",1,1,True,True


### 9.1 Answer 1 vs Answer 2 for each failure query

In [11]:
failure_results = []
for fq in FAILURE_QUERIES:
    print("=" * 110)
    print(f"[{fq['type']}]")
    failure_results.append(ask(fq["query"]))

[Keyword trap]


Q: Can a goal be scored directly from a kick-off?


,chunk,section,page,bi_rank,bi_score,ce_rank,ce_score,shift,preview
0,194,Law 16 - The Goal Kick,146,1,0.658,6,2.74,-5,If a player enters the penalty area before the ball is in play and fouls or is fouled by a
1,191,Law 16 - The Goal Kick,145,2,0.654,3,7.87,-1,"The Goal Kick A goal kick is awarded when the whole of the ball passes over the goal line,"
2,177,Law 14 - The Penalty Kick,135,3,0.624,2,8.07,1,The Penalty Kick A penalty kick is awarded if a player commits a direct free kick offence
3,124,Law 10 - Determining the Outcome of a Match,103,4,0.623,4,4.04,0,Determining the Outcome of a Match 1. Goal scored A goal is scored when the whole of the b
4,119,Law 8 - The Start and Restart of Play,97,5,0.616,1,8.20,4,"For every kick-off: • all players, except the player taking the kick-off, must be in their"
5,192,Law 16 - The Goal Kick,145,6,0.612,8,0.55,-2,2. Offences and sanctions If a player is unfairly delaying the taking of a goal kick for t
6,241,Law changes 2026/27 - Details of all Law changes,184,7,0.604,7,1.54,0,Explanation Text introduced to avoid delays at goal kicks. If a team is deliberately delay
7,182,Law 14 - The Penalty Kick,137,8,0.597,5,3.43,3,except for the following when play will be stopped and restarted with an indirect free kic
8,138,Law 11 - Offside,112,9,0.597,10,-0.92,-1,An attacking player may step or stay off the field of play not to be involved in active pl
9,193,Law 16 - The Goal Kick,145,10,0.590,9,0.33,1,"If, after the ball is in play, the kicker touches the ball again before it has touched ano"



Answer 1 (baseline top 3: [194, 191, 177]):
Not found in the provided context.

Answer 2 (re-ranked top 3: [119, 177, 191]):
Yes, a goal may be scored directly against the opponents from the kick-off [C1].
[Exception / corner case]


Q: What happens if the ball goes straight into the kicker's own goal from the kick-off?


...

,chunk,section,page,bi_rank,bi_score,ce_rank,ce_score,shift,preview
0,70,Law 3 - The Players,62,1,0.602,5,-2.21,-4,"If a player changes places with the goalkeeper without the referee’s permission, the refer"
1,147,Law 12 - Fouls and Misconduct,119,2,0.600,1,4.06,1,"3. Corner kick A corner kick is awarded if a goalkeeper, inside their penalty area, contro"
2,239,Law changes 2026/27 - Details of all Law changes,183,3,0.552,10,-5.88,-7,"Law 15 – The Throw-in (p. 141, 142) Additional text A throw-in is awarded to the opponents"
3,178,Law 14 - The Penalty Kick,135,4,0.543,9,-4.92,-5,The players other than the kicker and goalkeeper must be: • at least 9.15 m (10 yds) from
4,179,Law 14 - The Penalty Kick,136,5,0.539,8,-4.39,-3,The kicker must not play the ball again until it has touched another player. The penalty k
5,145,Law 12 - Fouls and Misconduct,118,6,0.539,3,1.20,3,"An indirect free kick is awarded if a goalkeeper, inside their penalty area, commits any o"
6,2,About the Laws,11,7,0.537,2,2.52,5,Poor participant behaviour and intentional delays that disrupt the flow of the game remain
7,144,Law 12 - Fouls and Misconduct,117,8,0.537,4,-1.45,4,No handball Handball No handball Handball Handball 2. Indirect free kick An indirect free
8,241,Law changes 2026/27 - Details of all Law changes,184,9,0.534,6,-3.65,3,Explanation Text introduced to avoid delays at goal kicks. If a team is deliberately delay
9,189,Law 15 - The Throw-in,141,10,0.531,7,-3.71,3,The thrower must not touch the ball again until it has touched another player. 2. Offences



Answer 1 (baseline top 3: [70, 147, 239]):
If a goalkeeper controls the ball with their hand(s)/arm(s) inside their penalty area for more than eight
seconds before releasing it, a corner kick is awarded to the opponents [C2].

Answer 2 (re-ranked top 3: [147, 2, 145]):
If a goalkeeper inside their penalty area controls the ball with their hand(s)/arm(s) for more than eight
seconds before releasing it, a corner kick is awarded to the opposition [C1, C2].
[Document-specific: struck-through (deleted) text extracted as live text]


Q: Is the kicker cautioned if both the penalty taker and the goalkeeper commit an offence at the same time?


,chunk,section,page,bi_rank,bi_score,ce_rank,ce_score,shift,preview
0,234,Law changes 2026/27 - Details of all Law changes,179,1,0.742,1,8.95,0,"• if the ball enters the goal from the kick, the kick is retaken • if the ball does not en"
1,185,Law 14 - The Penalty Kick,139,2,0.686,5,4.60,-3,Encroachment by defending and attacking player(s) Impact by both defending and attacking p
2,177,Law 14 - The Penalty Kick,135,3,0.671,9,2.59,-6,The Penalty Kick A penalty kick is awarded if a player commits a direct free kick offence
3,247,Law changes 2026/27 - Editorial changes,189,4,0.667,10,1.25,-6,Penalty kick Amended signal Image for corner kick changed to referee not blowing the whist
4,183,Law 14 - The Penalty Kick,137,5,0.666,2,7.13,3,"If the goalkeeper’s offence results in the kick being retaken, the goalkeeper is warned fo"
5,128,Law 10 - Determining the Outcome of a Match,106,6,0.660,3,6.63,3,"• if the ball enters the goal from the kick, the kick is retaken • if the ball does not en"
6,182,Law 14 - The Penalty Kick,137,7,0.658,7,3.96,0,except for the following when play will be stopped and restarted with an indirect free kic
7,179,Law 14 - The Penalty Kick,136,8,0.646,8,3.43,0,The kicker must not play the ball again until it has touched another player. The penalty k
8,220,Law changes 2026/27 - Outline summary,168,9,0.644,4,6.19,5,Law 10 – Determining the Outcome of a Match • Confirmation of circular no. 31 concerning a
9,249,Law changes 2026/27 - Editorial changes,191,10,0.644,6,4.57,4,Encroachment by defending and attacking player(s) Impact by both defending and attacking p



Answer 1 (baseline top 3: [234, 185, 177]):
No, the kicker is not cautioned if both the penalty taker and the goalkeeper commit an offence at the same
time, as the reference to the kicker being cautioned in this scenario has been removed [C1].

Answer 2 (re-ranked top 3: [234, 183, 128]):
No, the kicker is not cautioned because the reference to the kicker being cautioned for offending at the same
time as the goalkeeper was removed [C1].


## 10. Latency profiling
Each stage is timed over several runs after a warm-up. LLM time is reported separately because it dominates and is
identical in structure for both pipelines.

In [12]:
def profile_latency(queries, runs=5):
    for q in queries[:2]:                       # warm-up
        rerank(q, retrieve(q))
    rows = []
    for q in queries:
        for _ in range(runs):
            t0 = time.perf_counter(); cands = retrieve(q); t1 = time.perf_counter()
            rerank(q, cands);                            t2 = time.perf_counter()
            rows.append({"query": q, "bi_encoder_ms": (t1 - t0) * 1000, "cross_encoder_ms": (t2 - t1) * 1000})
    df = pd.DataFrame(rows)
    df["pipeline2_total_ms"] = df.bi_encoder_ms + df.cross_encoder_ms
    return df


lat = profile_latency([fq["query"] for fq in FAILURE_QUERIES])
latency_table = pd.DataFrame({
    "Pipeline 1 (bi-encoder retrieval)":       lat.bi_encoder_ms.describe()[["mean", "50%", "max"]],
    "Re-rank step only (cross-encoder, 10 pairs)": lat.cross_encoder_ms.describe()[["mean", "50%", "max"]],
    "Pipeline 2 (retrieval + re-rank)":         lat.pipeline2_total_ms.describe()[["mean", "50%", "max"]],
}).T.round(1).rename(columns={"50%": "median"})
latency_table.columns = [f"{c} (ms)" for c in latency_table.columns]
latency_table

,mean (ms),median (ms),max (ms)
Pipeline 1 (bi-encoder retrieval),9.8,9.4,24.3
"Re-rank step only (cross-encoder, 10 pairs)",267.5,275.2,344.0
Pipeline 2 (retrieval + re-rank),277.4,283.6,352.1


In [13]:
# LLM generation time, measured live (cache bypassed) for one query, for comparison with retrieval/re-ranking.
# Only 2 calls, to stay within the free-tier quota.
res = failure_results[0]
generate(res["query"], res["base_top3"], use_cache=False); t1 = llm_seconds[0]
generate(res["query"], res["rr_top3"], use_cache=False);   t2 = llm_seconds[0]
pd.DataFrame([
    {"stage": "Bi-encoder retrieval (top 10)", "ms": lat.bi_encoder_ms.mean()},
    {"stage": "Cross-encoder re-rank (10 pairs)", "ms": lat.cross_encoder_ms.mean()},
    {"stage": "LLM answer, baseline context", "ms": t1 * 1000},
    {"stage": "LLM answer, re-ranked context", "ms": t2 * 1000},
]).round(1)

,stage,ms
0,Bi-encoder retrieval (top 10),9.8
1,Cross-encoder re-rank (10 pairs),267.5
2,"LLM answer, baseline context",1218.2
3,"LLM answer, re-ranked context",967.0


In [14]:
# Why not cross-encode everything? Cost of scoring the whole corpus for one query:
q = FAILURE_QUERIES[0]["query"]
t0 = time.perf_counter()
cross_encoder.predict([(q, c["embed_text"]) for c in chunks], batch_size=32)
full_ce = time.perf_counter() - t0
print(f"Cross-encoder over all {len(chunks)} chunks: {full_ce * 1000:.0f} ms "
      f"(vs {lat.cross_encoder_ms.mean():.0f} ms for 10 candidates)")

Cross-encoder over all 301 chunks: 8227 ms (vs 268 ms for 10 candidates)


## 11. Try your own question

In [15]:
_ = ask("Can a player be offside if they receive the ball directly from a throw-in?")

Q: Can a player be offside if they receive the ball directly from a throw-in?


,chunk,section,page,bi_rank,bi_score,ce_rank,ce_score,shift,preview
0,137,Law 11 - Offside,111,1,0.691,1,9.50,0,3. No offence There is no offside offence if a player receives the ball directly from: • a
1,138,Law 11 - Offside,112,2,0.666,3,4.28,-1,An attacking player may step or stay off the field of play not to be involved in active pl
2,135,Law 11 - Offside,111,3,0.663,8,2.35,-5,"of instinctive stretching or jumping, or a movement that achieved limited contact/control"
3,131,Law 11 - Offside,109,4,0.659,5,3.38,-1,Offside 1. Offside position It is not an offence to be in an offside position. A player is
4,136,Law 11 - Offside,111,5,0.654,2,7.54,3,"blocks the opponent), the offence should be penalised under Law 12 • a player in an offsid"
5,133,Law 11 - Offside,110,6,0.633,4,3.80,2,or • clearly attempting to play a ball which is close when this action impacts on an oppon
6,134,Law 11 - Offside,110,7,0.630,6,3.19,1,• passing the ball to a team-mate; • gaining possession of the ball; or • clearing the bal
7,132,Law 11 - Offside,109,8,0.589,7,3.06,1,• second-last opponent or • last two opponents 2. Offside offence A player in an offside p
8,123,Law 9 - The Ball in and out of Play,101,9,0.586,9,0.27,0,2. Ball in play The ball is in play at all other times when it touches a match official an
9,75,Law 3 - The Players,64,10,0.577,10,-4.22,0,A player who crosses a boundary line as part of a playing movement does not commit an offe



Answer 1 (baseline top 3: [137, 138, 135]):
No, there is no offside offence if a player receives the ball directly from a throw-in [C1].

Answer 2 (re-ranked top 3: [137, 136, 138]):
No, there is no offside offence if a player receives the ball directly from a throw-in [C1, C2].
